In [ ]:
# Install dependencies
!pip install -q transformers==4.40.2 torch gradio pypdf accelerate sentencepiece

import gradio as gr
from transformers import pipeline
from pypdf import PdfReader

print("Loading models...")

# Load models
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

qa_model = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

print("Models loaded successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Models loaded successfully


In [ ]:
import gradio as gr
from pypdf import PdfReader

# --- HELPER FUNCTIONS ---

def extract_text(pdf_file):
    if pdf_file is None:
        return ""
    reader = PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text

def process_comprehensive_summary(text):
    if not text:
        return "No text to summarize."

    # Increase the scope: Divide text into chunks of 3000 chars
    # Summarize each chunk and combine for a "Bigger" summary
    chunk_size = 3000
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    summaries = []
    # Limit to first 3 chunks to avoid massive wait times, or remove limit for full doc
    for chunk in chunks[:3]:
        res = summarizer(chunk, max_length=150, min_length=40, do_sample=False)
        summaries.append(res[0]["summary_text"])

    return " ".join(summaries)

# --- MAIN AI FUNCTION ---

def legal_assistant_pro(pdf, text_input, question):
    # Determine source
    context = extract_text(pdf) if pdf else text_input

    if not context or len(context.strip()) == 0:
        return "Please provide a document or text.", "N/A"

    # 1. Bigger, more comprehensive summary
    full_summary = process_comprehensive_summary(context)

    # 2. Smart QA
    answer = "No question asked."
    if question:
        # We use a larger context window for QA to find specific details
        result = qa_model(question=question, context=context[:4000])
        answer = result["answer"]

    return full_summary, answer

# --- IMPROVED UI WITH BLOCKS ---

with gr.Blocks(theme=gr.themes.Soft(), title="Smart Legal Assistant v2") as demo:
    gr.Markdown("""
    # ⚖️ Smart Legal Assistant Pro
    *Analyze long legal judgments, extract summaries, and query specific clauses instantly.*
    """)

    with gr.Row():
        # Left Column: Inputs
        with gr.Column(scale=1):
            with gr.Tabs():
                with gr.TabItem("📄 Upload PDF"):
                    file_input = gr.File(label="Legal Document")
                with gr.TabItem("✍️ Paste Text"):
                    text_box = gr.Textbox(lines=12, label="Legal Text", placeholder="Paste clauses here...")

            question_input = gr.Textbox(label="Ask a specific question", placeholder="e.g., What is the notice period?")
            submit_btn = gr.Button("Analyze Document", variant="primary")

        # Right Column: Outputs
        with gr.Column(scale=1):
            summary_output = gr.Textbox(label="Comprehensive Summary", lines=10)
            answer_output = gr.Textbox(label="Direct Answer", lines=3)

    # Logic
    submit_btn.click(
        fn=legal_assistant_pro,
        inputs=[file_input, text_box, question_input],
        outputs=[summary_output, answer_output]
    )

demo.launch(share=True)

/tmp/ipykernel_438/1168982542.py:57: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Smart Legal Assistant v2") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://960489ff2b8f31b074.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=25a5a7af71f1f327a8b591984aa92352138af8411e98401a075f148f02a47ebb
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
!pip install -q transformers rouge-score

from transformers import pipeline
from rouge_score import rouge_scorer
import json
import numpy as np
from tqdm import tqdm
import pandas as pd

# 1. LOAD YOUR SUMMARIZATION MODEL HERE
# Example baseline:
model = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
)

# 2. LOAD DATASET
with open("legal_dataset.json", "r") as f:
    data = json.load(f)

docs = data["documents"]   # top-level key

texts = []
references = []

for d in docs:
    j = d.get("judgment", {})
    judgment_text = j.get("judgment_text") or j.get("judgmenttext")
    summary = j.get("summary")
    if judgment_text and summary:
        texts.append(judgment_text)
        references.append(summary)

print("Loaded samples:", len(texts))

if len(texts) == 0:
    raise ValueError("No samples found; check judgment_text/judgmenttext/summary keys.")

# 3. SUBSET FOR EVAL
max_samples = min(50, len(texts))    # increase later if needed
texts = texts[:max_samples]
references = references[:max_samples]

# 4. METRICS: ROUGE-L + token F1
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rouge_ls = []
f1_scores = []
predictions = []

for src, ref in tqdm(zip(texts, references), total=len(texts)):
    # Truncate very long judgments if needed
    src_in = src[:4000]

    result = model(
        src_in,
        max_new_tokens=200,
        do_sample=False
    )[0]["summary_text"]

    predictions.append(result)

    # ROUGE-L F1
    scores = scorer.score(ref, result)
    rouge_ls.append(scores["rougeL"].fmeasure)

    # Simple token-overlap F1
    ref_tokens = set(ref.lower().split())
    pred_tokens = set(result.lower().split())
    inter = ref_tokens & pred_tokens

    if not ref_tokens or not pred_tokens or not inter:
        f1_scores.append(0.0)
    else:
        precision = len(inter) / len(pred_tokens)
        recall = len(inter) / len(ref_tokens)
        f1 = 2 * precision * recall / (precision + recall)
        f1_scores.append(f1)

avg_rouge_l = float(np.mean(rouge_ls))
avg_f1 = float(np.mean(f1_scores))
print("\n\n")
print(f"Average ROUGE-L F1: {avg_rouge_l:.4f}")
print(f"Average token F1:   {avg_f1:.4f}")

# 5. SAVE DETAILED RESULTS
df = pd.DataFrame({
    "judgment_text": texts,
    "reference_summary": references,
    "pred_summary": predictions,
    "rougeL_f1": rouge_ls,
    "token_f1": f1_scores,
})
df.to_csv("legal_summarization_metrics.csv", index=False)
print("Saved metrics to legal_summarization_metrics.csv")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded samples: 200


100%|██████████| 50/50 [00:53<00:00,  1.08s/it]




Average ROUGE-L F1: 0.3334
Average token F1:   0.3800
Saved metrics to legal_summarization_metrics.csv


In [ ]:
!pip install -q transformers rouge-score tqdm pandas numpy

import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from rouge_score import rouge_scorer

# 1. INITIALIZE MODEL & DATA
model_name = "facebook/bart-large-cnn"
summarizer = pipeline("summarization", model=model_name, device=0) # Using GPU

with open("legal_dataset.json", "r") as f:
    data = json.load(f)

docs = data.get("documents", [])
texts, references = [], []

for d in docs:
    j = d.get("judgment", {})
    judgment_text = j.get("judgment_text") or j.get("judgmenttext")
    summary = j.get("summary")
    if judgment_text and summary:
        texts.append(judgment_text)
        references.append(summary)

# Subset for tuning (smaller subset to save time during tuning)
max_samples = min(20, len(texts))
tune_texts = texts[:max_samples]
tune_refs = references[:max_samples]

# 2. DEFINE HYPERPARAMETER GRID
# We will test combinations of beam search and length penalties
param_grid = [
    {"num_beams": 2, "length_penalty": 1.0, "no_repeat_ngram_size": 3},
    {"num_beams": 4, "length_penalty": 2.0, "no_repeat_ngram_size": 3},
    {"num_beams": 4, "length_penalty": 0.6, "no_repeat_ngram_size": 2},
]

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
best_score = -1
best_params = None

print("--- Starting Hyperparameter Tuning ---")

# 3. TUNING LOOP
for params in param_grid:
    current_scores = []
    print(f"Testing: {params}")

    for src, ref in zip(tune_texts, tune_refs):
        # Truncate input to model max length (usually 1024 for BART)
        res = summarizer(
            src[:4000],
            max_new_tokens=200,
            do_sample=False,
            **params
        )[0]["summary_text"]

        score = scorer.score(ref, res)["rougeL"].fmeasure
        current_scores.append(score)

    avg_score = np.mean(current_scores)
    print(f"Average ROUGE-L for this config: {avg_score:.4f}\n")

    if avg_score > best_score:
        best_score = avg_score
        best_params = params

print(f"--- Tuning Complete ---")
print(f"Best Parameters: {best_params} with Score: {best_score:.4f}")

# 4. FINAL RUN WITH BEST PARAMETERS
print("\nRunning final evaluation on full subset...")
final_max = min(50, len(texts))
final_texts = texts[:final_max]
final_refs = references[:final_max]

predictions = []
rouge_ls = []
f1_scores = []

for src, ref in tqdm(zip(final_texts, final_refs), total=len(final_texts)):
    result = summarizer(
        src[:4000],
        max_new_tokens=200,
        do_sample=False,
        **best_params
    )[0]["summary_text"]

    predictions.append(result)

    # Metrics
    r_score = scorer.score(ref, result)["rougeL"].fmeasure
    rouge_ls.append(r_score)

    # Token F1
    ref_tokens = set(ref.lower().split())
    pred_tokens = set(result.lower().split())
    inter = ref_tokens & pred_tokens
    if not ref_tokens or not pred_tokens or not inter:
        f1_scores.append(0.0)
    else:
        p, r = len(inter)/len(pred_tokens), len(inter)/len(ref_tokens)
        f1_scores.append(2 * p * r / (p + r))

# 5. SAVE & DISPLAY
avg_rouge_l = float(np.mean(rouge_ls))
avg_f1 = float(np.mean(f1_scores))

print(f"\nFinal Results with Best Params:")
print(f"Average ROUGE-L F1: {avg_rouge_l:.4f}")
print(f"Average token F1:   {avg_f1:.4f}")

df = pd.DataFrame({
    "judgment_text": final_texts,
    "reference_summary": final_refs,
    "pred_summary": predictions,
    "rougeL_f1": rouge_ls,
    "token_f1": f1_scores,
})
df.to_csv("legal_summarization_optimized.csv", index=False)
print("Saved to legal_summarization_optimized.csv")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


--- Starting Hyperparameter Tuning ---
Testing: {'num_beams': 2, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3}
Average ROUGE-L for this config: 0.3392

Testing: {'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3}
Average ROUGE-L for this config: 0.3411

Testing: {'num_beams': 4, 'length_penalty': 0.6, 'no_repeat_ngram_size': 2}
Average ROUGE-L for this config: 0.3358

--- Tuning Complete ---
Best Parameters: {'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3} with Score: 0.3411

Running final evaluation on full subset...


100%|██████████| 50/50 [00:46<00:00,  1.07it/s]


Final Results with Best Params:
Average ROUGE-L F1: 0.3334
Average token F1:   0.3800
Saved to legal_summarization_optimized.csv
